# Common Libraries and Global Variables

In [1]:
# Common Libraries
import os, sys
from dotenv import load_dotenv # requires python-dotenv

config_path = "../../../config" # explicit path to the config folder
sys.path.append(config_path)
from auth import acquire_bearer_token, StaticBearerTokenCredential


# Global libraries - recall to declare them as "global" in the functions where they are assigned
bearer_token_cognitiveservices = ""
bearer_token_azureai = ""
openai_endpoint = ""
deployment_name = ""
openai_api_version = ""
messages_system = ""
messages_user = ""
project_endpoint = ""
agentv2_id = ""
agentv2_instructions = ""
agentv2_question = ""

# Authentication & Environment setup

In [2]:
def authentication_and_environmentsetup():
    global bearer_token_cognitiveservices, bearer_token_azureai
    bearer_token_cognitiveservices, user_cognitiveservices = acquire_bearer_token(
        scope="https://cognitiveservices.azure.com/.default", # https://ai.azure.com/.default, https://cognitiveservices.azure.com/.default...
        auth_method="default") # default, cli, device

    bearer_token_azureai, _ = acquire_bearer_token(
        scope="https://ai.azure.com/.default", # https://ai.azure.com/.default, https://cognitiveservices.azure.com/.default...
        auth_method="default") # default, cli, device

    if not load_dotenv(f"{config_path}/credentials_my.env"):
        print("Environment variables not loaded, cell execution stopped")
    else:
        print("Environment variables have been loaded ;-)")

    print("Bearer token Cognitive Services:", bearer_token_cognitiveservices[:10], "...")
    print("Bearer token Azure AI:", bearer_token_azureai[:10], "...")
    print(f'User info Cognitive Services: {user_cognitiveservices["name"]}, upn: {user_cognitiveservices["upn"]}')
    # bearer_token_cognitiveservices['raw_claims']

authentication_and_environmentsetup()

Environment variables have been loaded ;-)
Bearer token Cognitive Services: eyJ0eXAiOi ...
Bearer token Azure AI: eyJ0eXAiOi ...
User info Cognitive Services: Mauro Minella, upn: mauro.minella@MngEnvMCAP883652.onmicrosoft.com


# Initialization

In [3]:
def init():
    global openai_endpoint, \
        deployment_name, \
        openai_api_version, \
        messages_system, \
        messages_user, \
        project_endpoint, \
        agentv2_id, \
        agentv2_instructions, \
        agentv2_question
    
    openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
    deployment_name = os.environ["MODEL_DEPLOYMENT_NAME"]
    openai_api_version = os.getenv("AZURE_OPENAI_API_VERSION")
    project_endpoint = os.getenv("AIF_STD_PROJECT_ENDPOINT")
    agentv2_id = os.getenv("PROMPT_AGENT_ID")
    agentv2_instructions = "You are a clever agent"
    agentv2_question = "How is the weather like today in New York?"
    messages_system = "You are an AI assistant that helps people find information"
    messages_user = "Tell me five good names for my new pizzeria"
    
init()

In [6]:
from agent_framework.azure import AzureAIAgentClient

chat_client=AzureAIAgentClient(
        project_endpoint=project_endpoint,
        model_deployment_name=deployment_name,
        credential=StaticBearerTokenCredential(bearer_token_cognitiveservices),
    )

In [18]:
from datetime import datetime
from zoneinfo import ZoneInfo

def get_local_date_time(iana_timezone: str = "Europe/Rome") -> str:
    """
    Get the current date and time for a given timezone.
    """
    try:
        tz = ZoneInfo(iana_timezone)
        current_time = datetime.now(tz)
        return (
            f"The current date and time in {iana_timezone} is "
            f"{current_time.strftime('%A, %B %d, %Y at %I:%M %p %Z')}"
        )
    except Exception as e:
        return (
            f"Error: Unable to get time for timezone '{iana_timezone}'. {str(e)}"
        )


from azure.ai.projects.models import FunctionTool
get_local_date_time_tool = FunctionTool(
    name="get_local_date_time",
    parameters={
        "type": "object",
        "properties": {
            "iana_timezone": {
                "type": "string",
                "description": (
                    "An IANA timezone string such as "
                    "'America/Los_Angeles', 'Europe/London', or 'Asia/Tokyo'."
                ),
            },
        },
        "required": ["iana_timezone"],
        "additionalProperties": False,
    },
    description="Return the current local date and time for a given IANA timezone.",
    strict=True,
    # function=get_local_date_time,  # <-- this binds your Python function
)


get_local_date_time("Europe/Rome")

'The current date and time in Europe/Rome is Tuesday, March 17, 2026 at 03:34 PM CET'

In [23]:
!uv pip install azure-ai-agentserver-agentframework

Audited 1 package in 40ms


In [26]:
from agent_framework import ChatAgent

ImportError: cannot import name 'ChatAgent' from 'agent_framework' (C:\Users\mauromi\source\git_repos\maf\python\maf01\.venv\Lib\site-packages\agent_framework\__init__.py)

In [20]:
# Create the agent with a local Python tool

from agent_framework import ChatAgent

agent = ChatAgent(
    chat_client=chat_client,
    instructions="You are a helpful assistant that can tell users the current date and time in any location. When a user asks about the time in a city or location, use the get_local_date_time tool with the appropriate IANA timezone string for that location.",
    tools=[get_local_date_time],
)

ImportError: cannot import name 'ChatAgent' from 'agent_framework' (C:\Users\mauromi\source\git_repos\maf\python\maf01\.venv\Lib\site-packages\agent_framework\__init__.py)

## MAF with local agent using `agent_framework` 
### Here we create a MAF **chat client** object (`AzureOpenAIChatClient`)...
### ...and then **transform** it into a local MAF **agent** (`Agent`)
- [The package `agent-framework`](https://pypi.org/project/agent-framework/) (version **1.0.0rc4** as of March 14th, 2026) contains all the Microsoft Agent Framework core and optional packages for building AI Agents with Python.<br/>
- The `AzureOpenAIChatClient` object is a **local** agent -part of the MAF runtime- that is an **orchestration wrapper** on the OpenAI model endpoint.<br/>
- Of course, this is **NOT** a **Foundry** `Persistent` or `Prompt` agent.<br/>
- Its `as_agent()` method transforms the ***chat client*** object into another MAF object that implements the MAF **Agent** interface, so that it can invoke the `run()` method to invoke the agent.

In [ ]:
async def create_maf_local_agent():
    from agent_framework.azure import AzureOpenAIChatClient # requires agent-framework-azure-ai
    from agent_framework import Message
    
    # create the MAF Chat Client object
    cc = AzureOpenAIChatClient(
            credential = StaticBearerTokenCredential(bearer_token_cognitiveservices),
            endpoint = openai_endpoint.rstrip("/") + "/",
            deployment_name = deployment_name,
            api_version = openai_api_version,
    )
    
    # convert the MAF Chat Client object into another object that implements the MAF Agent interface
    maf_local_agent = cc.as_agent() # python object that implements the Agent interface, so you can call agent.run() to get a response from the model
    
    
    # since "agent" implements the "Agent" interface, it can invoke the agent with the run() method
    response = await maf_local_agent.run( # this is asyncrhonous
        [
            Message(role="system", contents=messages_system),
            Message(role="user", contents=messages_user),
        ]
    )
    return response.text

# use x = await create_maf_local_agent() if running in a jupyter notebook
# use x = asyncio.run(create_maf_local_agent()) if running in a .py file
x = await create_maf_local_agent()
print(x)

# FOUNDRY

## Foundry V1 (aka Persistent, Classic) Agent
Options:
- REST API V1
- Package: [azure-ai-inference](https://pypi.org/project/azure-ai-inference/) version 1.0.0b9 as of 14/03/2026

## Foundry V2 (Prompt) Agents
Package: [azure-ai-projects](https://pypi.org/project/azure-ai-projects/) version 2.0.1 as of 14/03/2026

In [ ]:
# list V2 agent
from azure.ai.projects import AIProjectClient

foundryv2_project_client = AIProjectClient(
    endpoint=project_endpoint, 
    credential=StaticBearerTokenCredential(bearer_token_azureai))

for a in foundryv2_project_client.agents.list():
   print(a["id"], a.name)

In [ ]:
# retrieve an existing agent v2

existing_agentv2 = foundryv2_project_client.agents.get(agentv2_id)

existing_agentv2.as_dict()

In [ ]:
# create a new agent v2 using its definition

agent_definition = {
    "model": deployment_name,  # o il deployment che hai nel progetto
    "instructions": agent_instructions,
    "kind":"prompt"
    # opzionale: "tools": [...], "metadata": {...} ecc.
}

agent_version = foundryv2_project_client.agents.create_version(
    agent_name=f"{agentv2_id}2",
    definition=agent_definition,
    description=f"{agentv2_id}2",
)

agent_version

# MAF

## MAF with Microsoft Foundry `Classic` (V1, Persistent): [`agent_framework-azure-ai` package](https://pypi.org/project/agent-framework-azure-ai/)
### Here we create a MAF **service-based persistent agent (classic)** object (`AzureAIAgentClient`)
- [The package `agent-framework-azure-ai`](https://pypi.org/project/agent-framework/) (version **1.0.0rc4** as of March 14th, 2026) contains all the Microsoft Agent Framework packages for building Microsoft Foundry Native (V1) agents with Python.<br/>
- The `AzureOpenAIChatClient` object is a **local** agent -part of the MAF runtime- that is an **orchestration wrapper** on the OpenAI model endpoint.<br/>
- Its `as_agent()` method transforms the ***chat client*** object into another MAF object that implements the MAF **Agent** interface, so that it can invoke the `run()` method to invoke the agent.
- Of course, this is **NOT** a **Foundry** `Persistent` or `Prompt` agent

In [ ]:
from agent_framework.azure import AzureAIAgentClient, AzureAIAgentsProvider

chat_client = AzureAIAgentClient(
    credential=StaticBearerTokenCredential(bearer_token_azureai),
    project_endpoint=project_endpoint,
    model_deployment_name=deployment_name,
    agent_id=agent_id,
    # agent_name="deleteme001",
)

provider = AzureAIAgentsProvider(
    credential=StaticBearerTokenCredential(bearer_token_azureai),
    project_endpoint=project_endpoint,
    agents_client=chat_client
    # model_deployment_name=deployment_name,
    #agent_name="deleteme001",
)

async with provider:
    maf_agent = await provider.as_agent(  # <-- questo è MAF Agent
    print(type(chat_client), dir(chat_client))
    # resp = await maf_agent.run("Hi there")
    #print(resp.text)

In [ ]:
async def create_foundry_agent_v1():
    from agent_framework.azure import AzureAIAgentClient
    
    chat_client = AzureAIAgentClient(
        credential=StaticBearerTokenCredential(bearer_token_azureai),
        project_endpoint=project_endpoint,
        model_deployment_name=deployment_name,
        agent_name="deleteme001"
    )

    maf_agent = await chat_client.agents_client.get_agent(agent_id=agent_id)  # <-- questo è MAF Agent
    
    # maf_foundryv1_agent = chat_client.agents_client.create_agent(instructions=agent_instructions) # async method
    # print(dir(maf_foundryv1_agent))
    # return maf_foundryv1_agent
    
    # result = await maf_foundryv1_agent.run("Hello!")
    
    # result.text

x = await create_foundry_agent_v1()
type(x)

In [ ]:
from agent_framework.azure import AzureAIAgentClient

chat_client = AzureAIAgentClient(
    credential=StaticBearerTokenCredential(bearer_token_azureai),
    project_endpoint=project_endpoint,
    model_deployment_name=deployment_name,
)

maf_foundryv1_agent = await chat_client.agents_client.get_agent(agent_id=agent_id)

response = await maf_foundryv1_agent.run("Hi There")
response

In [ ]:
from agent_framework import ChatAgent
ChatAgent(chat_client =chat_client)

In [ ]:
import agent_framework.azure
dir(agent_framework.azure)

## Foundry V2
[The package `azure-ai-projects`](https://pypi.org/project/azure-ai-projects/) (version **2.0.1** as of March 14th, 2026) contains all the Microsoft Agent Framework packages for building Microsoft Foundry **Prompt** (V2) agents with Python, leveraging the [PromptAgentDefinition class](https://learn.microsoft.com/en-us/python/api/azure-ai-projects/azure.ai.projects.models.promptagentdefinition?view=azure-python-preview)